In [9]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm

In [10]:
model_name = 'intfloat/multilingual-e5-large'
lang = "fr"
batch_size = 16

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7882.89it/s]


In [12]:
collection_embeddings = torch.load('baseline/cache/collection_embeddings.pt')
collection_keys = np.load('baseline/cache/collection_keys.npy')
print(f"Loaded cached collection embeddings: {collection_embeddings.shape}")
print(f"Collection keys: {len(collection_keys)} entries")

Loaded cached collection embeddings: torch.Size([10000, 1024])
Collection keys: 10000 entries


### Load Collection Data

In [4]:
collection_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "collection")["collection"]
collection_keys = np.array(collection_data["pubkey"])

def create_text(example):
    text = f'passage: {example["title"]}. {example["venue"]}. {example["abstract"]}. {example["authors"]}'
    return {"text": text}

collection_data = collection_data.map(create_text)

def tokenize_with_padding(examples):
    return tokenizer(examples["text"], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

collection_data = collection_data.map(tokenize_with_padding, batched=True, remove_columns=collection_data.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

Map: 100%|██████████| 10000/10000 [00:04<00:00, 2156.72 examples/s]


### Embed Collection Data

In [5]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
    
embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(collection_data, batch_size=batch_size, num_workers=0)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        embeddings.append(batch_embeddings.detach().cpu())

collection_embeddings = torch.vstack(embeddings)
collection_embeddings = F.normalize(collection_embeddings, p=2, dim=1)

100%|██████████| 625/625 [1:46:55<00:00, 10.26s/it]   


### Load Query Data

In [13]:
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang)
train = data["train"]
train_labels = np.array(train["pubkey"])

dev = data["dev"]
dev_labels = np.array(dev["pubkey"])

In [14]:
import re

def clean_tweet(text):
    """Track 3 — tweet noise removal."""
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\d+/\d+\s*', '', text)
    text = re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)\']+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def process_text(example):
    text = clean_tweet(example["text"])
    text = f'query: {text}'
    return {"text": text}

train = train.map(process_text)
dev = dev.map(process_text)

def tokenize_with_padding(examples):
    return tokenizer(examples["text"], max_length=128, truncation=True, padding="max_length", return_tensors="pt")

train = train.map(tokenize_with_padding, batched=True, remove_columns=train.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
dev = dev.map(tokenize_with_padding, batched=True, remove_columns=dev.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

Map: 100%|██████████| 702/702 [00:00<00:00, 15219.85 examples/s]


In [15]:
from datasets import load_dataset as ld
raw = ld("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "fr")["train"].select(range(5))
for s in raw:
    print(f"RAW:   {s['text'][:130]}")
    print(f"CLEAN: {clean_tweet(s['text'])[:130]}")
    print("---")

RAW:   Le SRAS-CoV-2 peut infecter immédiatement un type spécialisé de cellules rénales. La trouvaille contribue à éclaircir pourquoi l'i
CLEAN: Le SRAS-CoV-2 peut infecter immédiatement un type spécialisé de cellules rénales. La trouvaille contribue à éclaircir pourquoi l'i
---
RAW:   Une étude dernièrement diffusée dans la revue « Food and Chemical Toxicology » a aussi associé les vaccins à ARNm à une hausse des
CLEAN: Une étude dernièrement diffusée dans la revue Food and Chemical Toxicology a aussi associé les vaccins à ARNm à une hausse des can
---
RAW:   Dans « Prolégomènes à une histoire des collections spéciales de l’Université de Montréal », Normand Trudel et Éric Bouchard suggèr
CLEAN: Dans Prolégomènes à une histoire des collections spéciales de l Université de Montréal , Normand Trudel et Éric Bouchard suggèrent
---
RAW:   Une récente étude démontre que le vieillissement épigénétique rapide est lié au danger de contamination au SARS‑CoV‑2 et aussi de 
CLEAN: Une récente étu

In [17]:
from torch import Tensor

def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

### Embed Query Data

In [18]:
train_embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(train, batch_size=batch_size, num_workers=0)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        train_embeddings.append(batch_embeddings.detach().cpu())

train_embeddings = torch.vstack(train_embeddings)
train_embeddings = F.normalize(train_embeddings, p=2, dim=1)

dev_embeddings = []
with torch.inference_mode():
  for batch in tqdm(DataLoader(dev, batch_size=batch_size, num_workers=0)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        dev_embeddings.append(batch_embeddings.detach().cpu())

dev_embeddings = torch.vstack(dev_embeddings)
dev_embeddings = F.normalize(dev_embeddings, p=2, dim=1)

100%|██████████| 44/44 [00:39<00:00,  1.12it/s]


### Calculate MRR@5 Score

In [19]:
def mrr_at_5(cos_sims, labels):
    # Get top-5 predictions
    top5_idxs = np.argsort(-cos_sims, axis=1)[:, :5]
    top5_keys = collection_keys[top5_idxs]
    
    # Compare labels to top-5 predictions
    labels_2d = labels[:, None]
    matches_mask = (top5_keys == labels_2d)

    # Get position of correct predictions
    positions = np.argmax(matches_mask, axis=1)
    positions[~matches_mask.any(axis=1)] = -1
    positions = positions.astype(int)

    # Get reciprocal rank
    rr = np.zeros_like(positions, dtype=float)
    mask = positions >= 0
    rr[mask] = 1.0 / (positions[mask] + 1)

    return float(rr.mean())

In [20]:
train_cos_sims = (train_embeddings @ collection_embeddings.T).numpy()
dev_cos_sims = (dev_embeddings @ collection_embeddings.T).numpy()

print("MRR@5 Train:", mrr_at_5(train_cos_sims, train_labels))
print("MRR@5 Dev:", mrr_at_5(dev_cos_sims, dev_labels))

MRR@5 Train: 0.4778945493409334
MRR@5 Dev: 0.461965811965812


In [21]:
import json, os
from datetime import datetime

os.makedirs('baseline/results', exist_ok=True)

new_train = 0.4778945493409334
new_dev = 0.461965811965812

results = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp1_clean_fr",
    "model": "intfloat/multilingual-e5-large",
    "language": "fr",
    "transformation": "rule_based_cleaning",
    "mrr5_train": new_train,
    "mrr5_dev": new_dev,
    "delta_train": new_train - 0.4697,
    "delta_dev": new_dev - 0.4584,
    "notes": "URL/@/# remove + emoji strip + numbering pattern removal. Note: SARS-CoV-2 splits incorrectly (Unicode dash issue, to fix)."
}

with open('baseline/results/results.jsonl', 'a') as f:
    f.write(json.dumps(results) + '\n')

print(json.dumps(results, indent=2))

{
  "timestamp": "2026-05-02T15:42:48.021936",
  "experiment": "exp1_clean_fr",
  "model": "intfloat/multilingual-e5-large",
  "language": "fr",
  "transformation": "rule_based_cleaning",
  "mrr5_train": 0.4778945493409334,
  "mrr5_dev": 0.461965811965812,
  "delta_train": 0.008194549340933388,
  "delta_dev": 0.0035658119658120158,
  "notes": "URL/@/# remove + emoji strip + numbering pattern removal. Note: SARS-CoV-2 splits incorrectly (Unicode dash issue, to fix)."
}


In [22]:
torch.save(train_embeddings, 'baseline/cache/train_embeddings_fr_clean.pt')
torch.save(dev_embeddings, 'baseline/cache/dev_embeddings_fr_clean.pt')
print("Saved cleaned FR query embeddings.")

Saved cleaned FR query embeddings.


In [23]:
import json, os
from datetime import datetime

os.makedirs('baseline/results', exist_ok=True)

new_train = 0.4778945493409334
new_dev = 0.461965811965812

results = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp1_clean_fr",
    "model": "intfloat/multilingual-e5-large",
    "language": "fr",
    "transformation": "rule_based_cleaning",
    "mrr5_train": new_train,
    "mrr5_dev": new_dev,
    "delta_train": new_train - 0.4697,
    "delta_dev": new_dev - 0.4584,
    "notes": "URL/@/# remove + emoji strip + numbering pattern removal."
}

with open('baseline/results/results.jsonl', 'a') as f:
    f.write(json.dumps(results) + '\n')

print(json.dumps(results, indent=2))

{
  "timestamp": "2026-05-02T15:43:33.478621",
  "experiment": "exp1_clean_fr",
  "model": "intfloat/multilingual-e5-large",
  "language": "fr",
  "transformation": "rule_based_cleaning",
  "mrr5_train": 0.4778945493409334,
  "mrr5_dev": 0.461965811965812,
  "delta_train": 0.008194549340933388,
  "delta_dev": 0.0035658119658120158,
  "notes": "URL/@/# remove + emoji strip + numbering pattern removal."
}
